In [2]:
import pandas as pd, sqlite3

conn = sqlite3.connect("../Dataset/olist.db")
orders = pd.read_sql("SELECT * FROM orders", conn)

date_cols = ["order_purchase_timestamp", "order_approved_at",
             "order_delivered_carrier_date", "order_delivered_customer_date",
             "order_estimated_delivery_date"]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

orders["is_delivered"] = (orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].notna()

orders["delay_days"] = (orders["order_delivered_customer_date"].dt.normalize()
                        - orders["order_estimated_delivery_date"].dt.normalize()).dt.days
orders["is_late"] = orders["delay_days"] > 0

delivered = orders[orders["is_delivered"]].copy()
print("Delivered orders:", len(delivered))
print("Late %:", round(delivered["is_late"].mean() * 100, 2))
print(delivered["delay_days"].describe())

Delivered orders: 96470
Late %: 6.77
count    96470.000000
mean       -11.875889
std         10.182105
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delay_days, dtype: float64


In [3]:
late = delivered[delivered["is_late"]]
print("Late orders:", len(late))
print("Average delay (late orders only):", round(late["delay_days"].mean(), 1), "days")
print("Median delay (late orders only):", late["delay_days"].median(), "days")

reviews = pd.read_sql("SELECT order_id, review_score, review_creation_date FROM order_reviews", conn)
print("Duplicate order_id in reviews:", reviews["order_id"].duplicated().sum())

reviews = reviews.sort_values("review_creation_date").drop_duplicates("order_id", keep="last")
merged = delivered.merge(reviews[["order_id", "review_score"]], on="order_id", how="left")
print(merged.groupby("is_late")["review_score"].agg(["count", "mean"]).round(2))

import os
os.makedirs("../Dataset/Cleaned", exist_ok=True)
orders.to_csv("../Dataset/Cleaned/orders_clean.csv", index=False)
delivered.to_csv("../Dataset/Cleaned/delivered_orders.csv", index=False)

Late orders: 6534
Average delay (late orders only): 10.6 days
Median delay (late orders only): 7.0 days
Duplicate order_id in reviews: 551
         count  mean
is_late             
False    89443  4.29
True      6381  2.27
